# Hidden Markov Model: Student Mood Analysis

In [1]:
import pandas as pd

## part 1: param learning

### load and explore the dataset

In [10]:
df = pd.read_csv('student_data.csv')
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 10 rows:")
print(df.head(10))

Total records: 100
Columns: ['StudentID', 'Day', 'Mood', 'ShirtColor']

First 10 rows:
   StudentID  Day Mood ShirtColor
0          1    1    H          R
1          1    2    H          R
2          1    3    S          B
3          1    4    S          B
4          1    5    H          R
5          1    6    H          G
6          1    7    H          R
7          1    8    S          B
8          1    9    S          G
9          1   10    S          B


In [13]:
# Get first day data for all students
first_day_data = df[df['Day'] == 1]

total_students = len(first_day_data)
mood_counts_first_day = first_day_data['Mood'].value_counts().to_dict()

print("Mood counts on first day:")
for mood, count in mood_counts_first_day.items():
    print(f"  {mood}: {count}")
print(f"\nTotal students: {total_students}")

Mood counts on first day:
  H: 3
  S: 2

Total students: 5


### Initial probabilities

In [16]:
initial_prob = {}

print("\nInitial Probability Distribution:")
print("Formula: P(M₁) = Count(M₁ on first day) / Total students")

for mood in ['H', 'S']:
    count = mood_counts_first_day.get(mood, 0)
    prob = count / total_students
    initial_prob[mood] = prob
    print(f"P({mood}) = {count}/{total_students} = {prob}")

print("\nInitial Probability Dictionary:")
print(initial_prob)


Initial Probability Distribution:
Formula: P(M₁) = Count(M₁ on first day) / Total students
P(H) = 3/5 = 0.6
P(S) = 2/5 = 0.4

Initial Probability Dictionary:
{'H': 0.6, 'S': 0.4}


In [17]:
from collections import defaultdict

transition_counts = defaultdict(lambda: defaultdict(int))
mood_counts = defaultdict(int)
for student_id in df['StudentID'].unique():
    student_data = df[df['StudentID'] == student_id].sort_values(by='Day')
    moods = student_data['Mood'].tolist()

    for i in range(len(moods) - 1):
        current_mood = moods[i]
        next_mood = moods[i+1]
        transition_counts[current_mood][next_mood] += 1
        mood_counts[current_mood] += 1

print('Transition Counts:')
for mood1 in ['H', 'S']:
    for mood2 in ['H', 'S']:
        count = transition_counts[mood1][mood2]
        print(f"  {mood1} -> {mood2}: {count}")
    print(f'total from {mood1} to {mood2}: {count}')

Transition Counts:
  H -> H: 36
  H -> S: 19
total from H to S: 19
  S -> H: 18
  S -> S: 22
total from S to S: 22


In [18]:
transition_counts

defaultdict(<function __main__.<lambda>()>,
            {'H': defaultdict(int, {'H': 36, 'S': 19}),
             'S': defaultdict(int, {'S': 22, 'H': 18})})

### Transition probabilities

In [20]:
transition_probab = defaultdict(dict)
print("\nTransition Matrix:")
print("Formula: P(M₂|M₁) = Count(M₁→M₂) / Count(M₁)")

for mood1 in ['H', 'S']:
    for mood2 in ['H', 'S']:
        count_transition = transition_counts[mood1][mood2]
        count_total = mood_counts[mood1]
        probab = count_transition / count_total
        transition_probab[mood1][mood2]=probab

print("transition probability dictionary:")
print(dict(transition_probab))
transition_df = pd.DataFrame(transition_probab).T
print("\nTransition Probability DataFrame:")
print(transition_df)



Transition Matrix:
Formula: P(M₂|M₁) = Count(M₁→M₂) / Count(M₁)
transition probability dictionary:
{'H': {'H': 0.6545454545454545, 'S': 0.34545454545454546}, 'S': {'H': 0.45, 'S': 0.55}}

Transition Probability DataFrame:
          H         S
H  0.654545  0.345455
S  0.450000  0.550000


### Emission probabilities

In [21]:
emission_counts = defaultdict(lambda: defaultdict(int))
total_mood_counts = defaultdict(int)

for _, row in df.iterrows():
    mood = row['Mood']
    color = row['ShirtColor']
    emission_counts[mood][color] += 1
    total_mood_counts[mood] += 1

In [22]:
emission_counts

defaultdict(<function __main__.<lambda>()>,
            {'H': defaultdict(int, {'R': 41, 'G': 16}),
             'S': defaultdict(int, {'B': 37, 'G': 6})})

In [26]:
emission_probab = defaultdict(dict)

print("\nEmission Matrix:")
print("Formula: P(C|M) = Count(C,M) / Count(M)")

for mood in ['H', 'S']:
    total = total_mood_counts[mood]
    for color in ['R', 'G', 'B']:
        count = emission_counts[mood][color]
        prob = count / total if total > 0 else 0.0
        emission_probab[mood][color] = prob
        print(f"P({color}|{mood}) = {count}/{total} = {prob:.4f}")



Emission Matrix:
Formula: P(C|M) = Count(C,M) / Count(M)
P(R|H) = 41/57 = 0.7193
P(G|H) = 16/57 = 0.2807
P(B|H) = 0/57 = 0.0000
P(R|S) = 0/43 = 0.0000
P(G|S) = 6/43 = 0.1395
P(B|S) = 37/43 = 0.8605


## part 2: inference

In [27]:
col = ["R","B","G"]

mood = ["H","S"]

sequences = []
probs      = []

for M1 in mood:
    for M2 in mood:
        for M3 in mood:
            
            # P(M1)
            p = initial_prob[M1]
            
            # emission of obs[0] at M1
            p *= emission_counts[M1][col[0]]

            # transition M1→M2 + emission of obs[1]
            p *= transition_probab[M1][M2]
            p *= emission_probab[M2][col[1]]

            # transition M2→M3 + emission of obs[2]
            p *= transition_probab[M2][M3]
            p *= emission_probab[M3][col[2]]

            sequences.append([M1,M2,M3])
            probs.append(p)

# zip everything
results = list(zip(sequences, probs))


In [28]:
# sort by probability (descending)
results.sort(key=lambda x:x[1], reverse=True)

print("Inference results for sequence:", col)
for seq, p in results:
    print(seq, round(p, 6))

best_seq, best_p = results[0]
print("\nMost likely:", best_seq, "| probability:", round(best_p,6))


Inference results for sequence: ['R', 'B', 'G']
['H', 'S', 'H'] 0.92367
['H', 'S', 'S'] 0.561183
['H', 'H', 'H'] 0.0
['H', 'H', 'S'] 0.0
['S', 'H', 'H'] 0.0
['S', 'H', 'S'] 0.0
['S', 'S', 'H'] 0.0
['S', 'S', 'S'] 0.0

Most likely: ['H', 'S', 'H'] | probability: 0.92367
